# Colab runner

End-to-end on a single T4. Everything persists to Google Drive, so a disconnect
costs minutes rather than the run.

Expect roughly 10 hours of GPU for the full grid at the published sample sizes.
Each stage resumes from its last checkpoint, so run it across several sittings.

**Runtime → Change runtime type → T4 GPU** before you start.

## 1. Install and mount

Restart the session when prompted, then skip this cell.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes datasets
!pip -q install "pandas==2.2.3"
print("\nRESTART SESSION NOW: Runtime > Restart session")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
os.environ["CASCADE_ROOT"] = "/content/drive/MyDrive/slm-cascade-routing"
pathlib.Path(os.environ["CASCADE_ROOT"]).mkdir(parents=True, exist_ok=True)

# Clone the replication package (or upload it) next to the Drive results folder.
REPO = "/content/slm-cascade-routing"
if not pathlib.Path(REPO).exists():
    # replace USER with your GitHub account, or upload the folder instead
    os.system(f"git clone https://github.com/USER/slm-cascade-routing {REPO}")
%cd /content/slm-cascade-routing

import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("results root:", os.environ["CASCADE_ROOT"])

## 2. Freeze the datasets

Run once. Writes four parquet files and a manifest with checksums to Drive.
Every later stage verifies its inputs against that manifest.

In [ ]:
!python scripts/00_freeze_datasets.py

## 3. Timing probe

Scoring cost scales with the label count, so the cheapest dataset is a poor
guide to the rest. This probe measures the real rate before you commit.

In [ ]:
import time, numpy as np
from src.config import DATASETS, DATASET_ORDER
from src.datasets import labels_of, load_frozen
from src.scoring import load_model, build_prefix, truncate, score_labels

tok, model = load_model("Qwen/Qwen3-1.7B", False)
rows = []
for ds in DATASET_ORDER:
    df = load_frozen(ds, 25); labels = labels_of(ds)
    ids = [tok(l, add_special_tokens=False, return_tensors="pt").input_ids for l in labels]
    t0 = time.time()
    for t in df.text:
        score_labels(model, tok, build_prefix(tok, truncate(tok, t, DATASETS[ds]["max_tokens"]), labels),
                     ids, model.device)
    per = (time.time()-t0)/len(df)
    hrs = per * DATASETS[ds]["n_eval"] / 3600
    rows.append((ds, len(labels), round(per,2), round(hrs,2)))
    print(f"{ds:10s} {len(labels):4d} labels  {per:5.2f}s/item  {hrs:5.2f} h for this combo")
print(f"\none small model over all four datasets: {sum(r[3] for r in rows):.1f} h")
print("the 14B in 4-bit runs roughly 4-5x slower than the 1.7B")

del model, tok
import gc, torch; gc.collect(); torch.cuda.empty_cache()

## 4. Small tiers

Both sizes over all four datasets. Re-run the cell after a disconnect and it
picks up where it stopped.

In [ ]:
!python -m src.run_inference --models qwen3-1.7b qwen3-4b

## 5. Large tier

Separate process so VRAM starts clean.

In [ ]:
!python -m src.run_inference --models qwen3-14b

## 6. Tables and figures

CPU only. Safe to run on a CPU runtime later.

In [ ]:
!python -m src.analysis
!python -m src.figures

## 7. Status board

Run any time to see what is finished.

In [ ]:
import pandas as pd, os
from pathlib import Path
from src.config import DATASETS, PREDS
rows=[]
for f in sorted(Path(PREDS).glob("*.parquet")):
    if "scores" in f.name: continue
    tag, ds = f.stem.split("__")
    p = pd.read_parquet(f)
    target = DATASETS[ds]["n_eval"]
    rows.append({"model": tag, "dataset": ds, "done": len(p), "target": target,
                 "complete": len(p) >= target,
                 "accuracy": round((p.pred == p.gold).mean(), 3)})
print(pd.DataFrame(rows).to_string(index=False) if rows else "nothing yet")